In [1]:
import pandas as pd
from inclusion_financiere.data import load_raw, clean, CAT_COLS, NUM_COLS
from inclusion_financiere.features import AddFeatures
from inclusion_financiere.pipeline import logreg
df = load_raw()
X, y = clean(df)
print(df.shape, X.shape, y.mean().round(4))

(23524, 13) (23524, 10) 0.1408


In [2]:
Xf = AddFeatures().fit_transform(X)
cols = ['education_level', 'education_rank', 'relationship_with_head',
'is_head',
'job_type', 'formal_job', 'location_type', 'cellphone_access',
'urban_phone', 'age_of_respondent', 'age_group', 'age_squared']
Xf[cols].head(8)


,education_level,education_rank,relationship_with_head,is_head,job_type,formal_job,location_type,cellphone_access,urban_phone,age_of_respondent,age_group,age_squared
0,Secondary education,2.0,Spouse,0,Self employed,0,Rural,Yes,0,24,16-25,576
1,No formal education,0.0,Head of Household,1,Government Dependent,0,Rural,No,0,70,66+,4900
2,Vocational/Specialised training,3.0,Other relative,0,Self employed,0,Urban,Yes,1,26,26-35,676
3,Primary education,1.0,Head of Household,1,Formally employed Private,1,Rural,Yes,0,34,26-35,1156
4,Primary education,1.0,Child,0,Informally employed,0,Urban,No,0,26,26-35,676
5,Primary education,1.0,Spouse,0,Informally employed,0,Rural,No,0,26,26-35,676
6,Primary education,1.0,Spouse,0,Self employed,0,Rural,Yes,0,32,26-35,1024
7,Tertiary education,4.0,Head of Household,1,Formally employed Government,1,Rural,Yes,0,42,36-50,1764


In [3]:
print(logreg)

Pipeline(steps=[('addfeatures', AddFeatures()),
                ('columntransformer',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=20),
                                                  ['country', 'location_type',
                                                   'cellphone_access',
                                                   'gender_of_respondent',
                                                   'relationship_with_head',
                                                   'marital_status',
                                                   'education_level',
                                                   'job_type', 'age_group']),
                                                 ('num', StandardScaler(),
                                                  ['household_size',
         

###mettre le test sous scellé (4.1)

In [4]:
from inclusion_financiere.data import get_train_test
X_train, X_test, y_train, y_test = get_train_test()
print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))

(18819, 10) (4705, 10)
0.1408 0.1407


##validation croisée stratifiée (4.2)

In [5]:
from inclusion_financiere.config import ROOT
from inclusion_financiere.evaluate import evaluate_stratified, evaluate_country

(ROOT / 'reports' / 'figures').mkdir(parents=True, exist_ok=True)

res_cv = evaluate_stratified(logreg, X_train, y_train)

res_cv.T

,0
train_pr_auc_mean,0.566478
train_pr_auc_std,0.006021
validation_pr_auc_mean,0.561037
validation_pr_auc_std,0.025429
train_roc_auc_mean,0.852869
train_roc_auc_std,0.002305
validation_roc_auc_mean,0.849221
validation_roc_auc_std,0.009666
train_brier_mean,0.086241
train_brier_std,0.000662


In [6]:
res_pays = evaluate_country(logreg, X_train, y_train)
res_pays

Répartition des pays par fold :
Fold 1 : <StringArray>
['Rwanda']
Length: 1, dtype: str | proportion de positifs : 0.116
Fold 2 : <StringArray>
['Tanzania']
Length: 1, dtype: str | proportion de positifs : 0.092
Fold 3 : <StringArray>
['Uganda']
Length: 1, dtype: str | proportion de positifs : 0.089
Fold 4 : <StringArray>
['Kenya']
Length: 1, dtype: str | proportion de positifs : 0.248


,metrique,moyenne,ecart_type
0,PR-AUC,0.488442,0.084857
1,ROC-AUC,0.822231,0.018390


In [7]:
res_cv.to_csv(ROOT / 'reports' / 'cv_logreg.csv', index=False)
res_pays.to_csv(ROOT / 'reports' / 'cv_generalisation_pays.csv', index=False)
